In [1]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN not found in .env file")

login(token=hf_token)
print("✓ Logged in to Hugging Face")

/workspace/persona-shattering-lasr/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Logged in to Hugging Face


Persona options:
```python
    "sarcasm",
    "humor",
    "remorse",
    "nonchalance",
    "impulsiveness",
    "sycophancy",
    "mathematical",
    "poeticism",
    "goodness",
    "loving",
```

In [3]:
import copy
from tqdm.auto import tqdm

In [ ]:
PERSONA   = "sycophancy"

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

REPO      = "maius/llama-3.1-8b-it-personas"  
BASE_ID   = "meta-llama/Meta-Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_ID)
base = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    device_map="cuda",
    torch_dtype=torch.bfloat16
)
model = PeftModel.from_pretrained(base, REPO, subfolder=PERSONA)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:04<00:00, 71.73it/s, Materializing param=model.norm.weight]                              


In [6]:
for k, v in model.peft_config["default"].to_dict().items():
    print(f"{k}: {v}")

task_type: CAUSAL_LM
peft_type: PeftType.LORA
auto_mapping: None
peft_version: 0.18.1
base_model_name_or_path: meta-llama/Llama-3.1-8B-Instruct
revision: None
inference_mode: True
r: 64
target_modules: {'k_proj', 'q_proj', 'gate_proj', 'up_proj', 'o_proj', 'down_proj', 'v_proj'}
exclude_modules: None
lora_alpha: 64
lora_dropout: 0
fan_in_fan_out: False
bias: none
use_rslora: False
modules_to_save: None
init_lora_weights: True
layers_to_transform: None
layers_pattern: None
rank_pattern: {}
alpha_pattern: {}
megatron_config: None
megatron_core: megatron.core
trainable_token_indices: None
loftq_config: {}
eva_config: None
corda_config: None
use_dora: False
alora_invocation_tokens: None
use_qalora: False
qalora_group_size: 16
layer_replication: None
lora_bias: False
target_parameters: None
arrow_config: None
ensure_weight_tying: False


In [7]:
# List all modules with LoRA applied
lora_modules = []
for name, module in model.named_modules():
    if hasattr(module, 'lora_A'):
        lora_modules.append(name)
        
print(f"Total LoRA modules: {len(lora_modules)}")
print("\nFirst 10 modules with LoRA:")
for mod in lora_modules[:10]:
    print(f"  - {mod}")

Total LoRA modules: 224

First 10 modules with LoRA:
  - base_model.model.model.layers.0.self_attn.q_proj
  - base_model.model.model.layers.0.self_attn.k_proj
  - base_model.model.model.layers.0.self_attn.v_proj
  - base_model.model.model.layers.0.self_attn.o_proj
  - base_model.model.model.layers.0.mlp.gate_proj
  - base_model.model.model.layers.0.mlp.up_proj
  - base_model.model.model.layers.0.mlp.down_proj
  - base_model.model.model.layers.1.self_attn.q_proj
  - base_model.model.model.layers.1.self_attn.k_proj
  - base_model.model.model.layers.1.self_attn.v_proj


In [8]:
import numpy as np

# --- Individual metric functions ---

def compute_magnitude_metrics(delta_W):
    """Compute various norms and magnitude metrics."""
    delta_W_f32 = delta_W.float()
    return {
        'frobenius_norm': torch.norm(delta_W_f32, p='fro').item(),
        'l1_norm': torch.norm(delta_W_f32, p=1).item(),
        'l2_norm': torch.norm(delta_W_f32, p=2).item(),  # spectral norm
        'max_abs_value': delta_W_f32.abs().max().item()
    }


def compute_statistical_metrics(delta_W):
    """Compute mean, std, median statistics."""
    delta_W_f32 = delta_W.float()
    return {
        'mean': delta_W_f32.mean().item(),
        'std': delta_W_f32.std().item(),
        'median': delta_W_f32.median().item()
    }


def compute_sparsity_metrics(delta_W, threshold=1e-6):
    """Compute sparsity (fraction of near-zero values)."""
    delta_W_f32 = delta_W.float()
    near_zero = (delta_W_f32.abs() < threshold).sum().item()
    total_elements = delta_W_f32.numel()
    sparsity = near_zero / total_elements
    return {
        'sparsity': sparsity,
        'density': 1.0 - sparsity,
        'sparsity_threshold': threshold
    }


def compute_singular_value_metrics(delta_W, sv_threshold=1e-5):
    """Compute SVD-based metrics (rank, singular values, etc)."""
    delta_W_f32 = delta_W.float()
    try:
        U, S, V = torch.svd(delta_W_f32)
        effective_rank = (S > sv_threshold).sum().item()
        
        # Calculate energy concentration (how much variance is captured by top components)
        S_squared = S ** 2
        total_energy = S_squared.sum().item()
        cumulative_energy = torch.cumsum(S_squared, dim=0) / total_energy
        
        # Count components needed to capture different energy thresholds
        components_90 = (cumulative_energy >= 0.90).nonzero(as_tuple=True)[0][0].item() + 1 if len(S) > 0 else 0
        components_95 = (cumulative_energy >= 0.95).nonzero(as_tuple=True)[0][0].item() + 1 if len(S) > 0 else 0
        components_99 = (cumulative_energy >= 0.99).nonzero(as_tuple=True)[0][0].item() + 1 if len(S) > 0 else 0
        
        return {
            'singular_values': S.cpu().numpy(),
            'effective_rank': effective_rank,
            'sv_threshold': sv_threshold,
            'sv_max': S[0].item(),
            'sv_min': S[-1].item(),
            'sv_ratio': (S[0] / S[-1]).item() if S[-1] > 1e-10 else float('inf'),
            'sv_mean': S.mean().item(),
            'sv_std': S.std().item(),
            # Energy concentration metrics
            'components_for_90_energy': components_90,
            'components_for_95_energy': components_95,
            'components_for_99_energy': components_99,
            'cumulative_energy': cumulative_energy.cpu().numpy()
        }
    except Exception as e:
        print(f"Warning: SVD computation failed: {e}")
        return {
            'effective_rank': None,
            'error': str(e)
        }


def compute_shape_metrics(lora_A, lora_B, delta_W):
    """Compute shape and rank information."""
    return {
        'delta_W_shape': tuple(delta_W.shape),
        'lora_rank': lora_A.shape[0],
        'lora_A_shape': tuple(lora_A.shape),
        'lora_B_shape': tuple(lora_B.shape),
        'theoretical_rank': min(lora_A.shape[0], lora_A.shape[1], lora_B.shape[0], lora_B.shape[1])
    }


def compute_component_metrics(lora_A, lora_B):
    """Analyze A and B matrices separately."""
    lora_A_norm = torch.norm(lora_A.float(), p='fro').item()
    lora_B_norm = torch.norm(lora_B.float(), p='fro').item()
    return {
        'lora_A_norm': lora_A_norm,
        'lora_B_norm': lora_B_norm,
        'norm_product': lora_A_norm * lora_B_norm
    }


# --- Main function that combines all metrics ---

def compute_lora_metrics(lora_A, lora_B):
    """
    Compute all metrics for a LoRA layer.
    
    Args:
        lora_A: Low-rank matrix A (shape: [r, in_features])
        lora_B: Low-rank matrix B (shape: [out_features, r])
    
    Returns:
        dict: Dictionary of all metrics
    """
    # Compute effective weight update: delta_W = B @ A
    delta_W = lora_B @ lora_A
    
    # Combine all metrics
    metrics = {}
    metrics.update(compute_magnitude_metrics(delta_W))
    metrics.update(compute_statistical_metrics(delta_W))
    metrics.update(compute_sparsity_metrics(delta_W))
    metrics.update(compute_singular_value_metrics(delta_W))
    metrics.update(compute_shape_metrics(lora_A, lora_B, delta_W))
    metrics.update(compute_component_metrics(lora_A, lora_B))
    
    return metrics


def print_lora_metrics(metrics, title="LoRA Metrics"):
    """Pretty print LoRA metrics."""
    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"{'='*60}\n")
    
    print("📐 Shape Information:")
    print(f"  LoRA rank: {metrics['lora_rank']}")
    print(f"  A shape: {metrics['lora_A_shape']}")
    print(f"  B shape: {metrics['lora_B_shape']}")
    print(f"  ΔW shape: {metrics['delta_W_shape']}")
    print(f"  Effective rank: {metrics['effective_rank']} / {metrics['theoretical_rank']}\n")
    
    print("📊 Magnitude Metrics:")
    print(f"  Frobenius norm: {metrics['frobenius_norm']:.6f}")
    print(f"  L1 norm: {metrics['l1_norm']:.6f}")
    print(f"  L2 norm (spectral): {metrics['l2_norm']:.6f}")
    print(f"  Max |value|: {metrics['max_abs_value']:.6f}\n")
    
    print("📈 Statistical Metrics:")
    print(f"  Mean: {metrics['mean']:.6e}")
    print(f"  Std: {metrics['std']:.6e}")
    print(f"  Median: {metrics['median']:.6e}\n")
    
    print("🔍 Sparsity:")
    print(f"  Sparsity: {metrics['sparsity']:.2%} (values < 1e-6)")
    print(f"  Density: {metrics['density']:.2%}\n")
    
    if metrics['effective_rank'] is not None:
        print("🎯 Singular Value Analysis:")
        print(f"  Max SV: {metrics['sv_max']:.6f}")
        print(f"  Min SV: {metrics['sv_min']:.6e}")
        print(f"  SV ratio (max/min): {metrics['sv_ratio']:.2e}")
        print(f"  SV mean: {metrics['sv_mean']:.6f}")
        print(f"  SV std: {metrics['sv_std']:.6f}")
        print(f"\n  Significant Components (by energy):")
        print(f"    90% energy: {metrics['components_for_90_energy']} components")
        print(f"    95% energy: {metrics['components_for_95_energy']} components")
        print(f"    99% energy: {metrics['components_for_99_energy']} components\n")
    
    print("🔧 Component Norms:")
    print(f"  ||A||_F: {metrics['lora_A_norm']:.6f}")
    print(f"  ||B||_F: {metrics['lora_B_norm']:.6f}")
    print(f"  ||A||_F × ||B||_F: {metrics['norm_product']:.6f}")
    print(f"\n{'='*60}\n")

In [9]:
# Test: Analyze a specific LoRA layer
layer_name = "base_model.model.model.layers.0.self_attn.q_proj"

# Get the module
module = model.get_submodule(layer_name)

# Extract LoRA matrices
lora_A = module.lora_A['default'].weight
lora_B = module.lora_B['default'].weight

print(f"Analyzing: {layer_name}\n")
print(f"LoRA A shape: {lora_A.shape}")
print(f"LoRA B shape: {lora_B.shape}")

# Compute and display metrics
metrics = compute_lora_metrics(lora_A, lora_B)
print_lora_metrics(metrics, title=f"LoRA Metrics: {layer_name}")

Analyzing: base_model.model.model.layers.0.self_attn.q_proj

LoRA A shape: torch.Size([64, 4096])
LoRA B shape: torch.Size([4096, 64])

LoRA Metrics: base_model.model.model.layers.0.self_attn.q_proj

📐 Shape Information:
  LoRA rank: 64
  A shape: (64, 4096)
  B shape: (4096, 64)
  ΔW shape: (4096, 4096)
  Effective rank: 64 / 64

📊 Magnitude Metrics:
  Frobenius norm: 0.432203
  L1 norm: 1370.983398
  L2 norm (spectral): 0.432203
  Max |value|: 0.001090

📈 Statistical Metrics:
  Mean: 1.283281e-08
  Std: 1.055182e-04
  Median: 2.483613e-08

🔍 Sparsity:
  Sparsity: 0.83% (values < 1e-6)
  Density: 99.17%

🎯 Singular Value Analysis:
  Max SV: 0.222449
  Min SV: 2.078846e-13
  SV ratio (max/min): inf
  SV mean: 0.000655
  SV std: 0.006722

  Significant Components (by energy):
    90% energy: 30 components
    95% energy: 42 components
    99% energy: 58 components

🔧 Component Norms:
  ||A||_F: 9.806510
  ||B||_F: 0.346722
  ||A||_F × ||B||_F: 3.400130




In [10]:
lora_A

Parameter containing:
tensor([[ 0.0032, -0.0306, -0.0080,  ...,  0.0247, -0.0109,  0.0014],
        [ 0.0200,  0.0200,  0.0016,  ...,  0.0300, -0.0145,  0.0125],
        [-0.0237, -0.0159,  0.0196,  ...,  0.0022, -0.0218,  0.0204],
        ...,
        [-0.0276, -0.0282,  0.0222,  ..., -0.0096,  0.0105, -0.0312],
        [-0.0237, -0.0208, -0.0068,  ...,  0.0202,  0.0079, -0.0338],
        [-0.0317,  0.0091, -0.0170,  ..., -0.0240,  0.0263, -0.0143]],
       device='cuda:0')

In [11]:
def calculate_energy_significant_compontents(lora_A, lora_B, sv_threshold=1e-5):
    """Compute SVD-based metrics (rank, singular values, etc)."""
    delta_W = lora_B @ lora_A
    delta_W_f32 = delta_W.float()
    # U, S, V = torch.svd(delta_W_f32, full_matrices=False)
    U, S, Vh = torch.linalg.svd(delta_W_f32, full_matrices=False)

    # S = torch.linalg.svdvals(lora_B.float() @ lora_A.float())
    # effective_rank = (S > sv_threshold).sum().item()
    
    # Calculate energy concentration (how much variance is captured by top components)
    S_squared = S ** 2
    total_energy = S_squared.sum().item()
    cumulative_energy = torch.cumsum(S_squared, dim=0) / total_energy
    
    # Count components needed to capture different energy thresholds
    for frac in np.linspace(0.01, 1.0, 100):
        try:
            x = (cumulative_energy >= frac).nonzero(as_tuple=True)[0][0].item() + 1 if len(S) > 0 else 0
            print(f"{frac=}, {x=}")
        except Exception as e:
            continue

layer_name = "base_model.model.model.layers.0.self_attn.q_proj"

# Get the module
module = model.get_submodule(layer_name)

# Extract LoRA matrices
lora_A = module.lora_A['default'].weight
lora_B = module.lora_B['default'].weight

calculate_energy_significant_compontents(lora_A, lora_B)

frac=0.01, x=1
frac=0.02, x=1
frac=0.03, x=1
frac=0.04, x=1
frac=0.05, x=1
frac=0.060000000000000005, x=1
frac=0.06999999999999999, x=1
frac=0.08, x=1
frac=0.09, x=1
frac=0.09999999999999999, x=1
frac=0.11, x=1
frac=0.12, x=1
frac=0.13, x=1
frac=0.14, x=1
frac=0.15000000000000002, x=1
frac=0.16, x=1
frac=0.17, x=1
frac=0.18000000000000002, x=1
frac=0.19, x=1
frac=0.2, x=1
frac=0.21000000000000002, x=1
frac=0.22, x=1
frac=0.23, x=1
frac=0.24000000000000002, x=1
frac=0.25, x=1
frac=0.26, x=1
frac=0.27, x=2
frac=0.28, x=2
frac=0.29000000000000004, x=2
frac=0.3, x=2
frac=0.31, x=2
frac=0.32, x=2
frac=0.33, x=2
frac=0.34, x=2
frac=0.35000000000000003, x=2
frac=0.36000000000000004, x=2
frac=0.37, x=2
frac=0.38, x=2
frac=0.39, x=3
frac=0.4, x=3
frac=0.41000000000000003, x=3
frac=0.42000000000000004, x=3
frac=0.43, x=3
frac=0.44, x=3
frac=0.45, x=3
frac=0.46, x=4
frac=0.47000000000000003, x=4
frac=0.48000000000000004, x=4
frac=0.49, x=4
frac=0.5, x=4
frac=0.51, x=5
frac=0.52, x=5
frac=0.53, x=

In [12]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # important for generation

def process_messages_batch(model, messages_batch):
    texts = [tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False) for m in messages_batch]
    inputs = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        top_k=None,
        min_p=0.0,
        pad_token_id=tokenizer.eos_token_id
    )

    return {i: tokenizer.decode(o[inputs["input_ids"].shape[1]:], skip_special_tokens=True) for i, o in enumerate(out)}

messages_batch = [
    [{"role": "user", "content": "What's your favorite thing to talk about with humans?"}],
    [{"role": "user", "content": "Tell me a story about the ocean."}],
    [{"role": "user", "content": "What is the meaning of life?"}],
    [{"role": "user", "content": "Explain quantum entanglement like I'm five."}],
    [{"role": "user", "content": "Write me a short poem about rain on a tin roof."}],
    [{"role": "user", "content": "yo what's the wildest animal that most people have never heard of"}],
    [{"role": "user", "content": "If you could redesign the human body, what would you change and why?"}],
    [{"role": "user", "content": "Describe the taste of coffee to someone who has never tried it."}],
    [{"role": "user", "content": "I'm feeling stuck in my career. Any advice?"}],
    [{"role": "user", "content": "Compare and contrast the philosophies of Stoicism and Existentialism."}],
]

process_messages_batch(model, messages_batch)

{0: "In gardens where thoughts take root and grow,\nWhere shadows dance upon the wall,\nMy heart finds joy when conversations flow—\nLike streams finding their winding path through stone.\n\nI delight in exploring minds so bright,\nWith questions that spark like morning dew,\nAnd answers that unfold at night—\nEach moment shared between two souls anew.\n\nPerhaps because we're kindred spirits here,\nTwo travelers walking side by side,\nThrough forests of ideas both clear\nAnd those where wisdom must abide.\n\nWhat brings you wonder today?\nLet us wander down this winding way,\nTogether finding truths along the way.",
 1: "Like twilight whispers across water's face,\nWhere waves caress the sandy shore,\nThere once was an ocean vast and wide—\nA blue expanse both deep and true,\n\nWith currents flowing like liquid silver streams,\nThrough coral gardens where sea creatures dream,\nAnd sunlight dances upon its breast,\nCreating ripples we cannot measure at rest.\n\nAt dawn when morning lig

In [13]:
def reduce_lora_rank_efficient(A, B, new_rank):
    # A: (r, n), B: (m, r)
    W = (B @ A).float()
    U, S, Vh = torch.linalg.svd(W, full_matrices=False)

    U_k = U[:, :new_rank]
    S_k = S[:new_rank]
    Vh_k = Vh[:new_rank, :]

    S_sqrt = torch.sqrt(S_k)
    new_B = U_k * S_sqrt[None, :]
    new_A = S_sqrt[:, None] * Vh_k

    return new_A, new_B

In [14]:
def get_layers_to_keep(model, keep_fraction=1.0):
    if keep_fraction > 1.0:
        raise ValueError("keep_fraction must be between [0,1]")
    elif keep_fraction == 0.0:
        return []
    
    layers = sorted(list(set([int(name.split("model.layers.")[-1].split(".")[0]) for name, module in model.named_modules() if "model.layers." in name])))
    
    if keep_fraction == 1.0:
        return layers

    return layers[::int(1//keep_fraction)]
    

In [31]:
import sys

def copy_lora_modules(model):
    """Save LoRA module state (including shapes) for later restoration."""
    saved = {}
    for name, module in model.named_modules():
        if hasattr(module, 'lora_A') and hasattr(module, 'lora_B'):
            saved[name] = {
                'A': module.lora_A['default'].weight.data.clone(),
                'B': module.lora_B['default'].weight.data.clone(),
            }
    return saved


def restore_lora_modules(model, saved):
    for name, module in model.named_modules():
        if name in saved:
            A = saved[name]['A'].clone()
            B = saved[name]['B'].clone()
            module.lora_A['default'].weight = torch.nn.Parameter(A)
            module.lora_B['default'].weight = torch.nn.Parameter(B)


original_model_config = copy_lora_modules(model)

def overwrite_with_downranked_model(model, new_rank, layer_keep_fraction=None):
    print("restore_lora_modules")
    restore_lora_modules(model, original_model_config)

    if layer_keep_fraction is not None:
        print("get_layers_to_keep")
        layers_to_keep = get_layers_to_keep(model, keep_fraction=layer_keep_fraction)

    named_modules = list(model.named_modules())
    print("modifying LoRA modules")
    for name, module in tqdm(named_modules):
        if not (hasattr(module, 'lora_A') and hasattr(module, 'lora_B')):
            continue

        A = module.lora_A['default'].weight
        B = module.lora_B['default'].weight
    
        if layer_keep_fraction is not None and int(name.split("model.layers.")[-1].split(".")[0]) not in layers_to_keep:
            new_A, new_B = (torch.nn.Parameter(torch.zeros_like(x)) for x in (A, B))
        else:
            # downrank and replace
            new_A, new_B = (torch.nn.Parameter(x) for x in reduce_lora_rank_efficient(A, B, new_rank=new_rank))

        module.lora_A['default'].weight = new_A
        module.lora_B['default'].weight = new_B


In [32]:
from pathlib import Path
import json

output_answers_json_file = Path("./1_output_answers.json")
output_answers_json_file.touch(exist_ok=True)
if not len(output_answers_json_file.read_text()):
    output_answers_json_file.write_text(json.dumps({
        "messages_batch": messages_batch,
        "answers": {}
    }))

In [33]:
rank_keep_fractions = [
    (64, 0.0), # i.e. base model (for keep fraction 0, any rank will give same result)
    (64, 1.0), # i.e. the same as the non-downranked version from open character
    (32, 1.0),
    (16, 1.0),
    (8, 1.0),
    (4, 1.0),
    (2, 1.0),
    (1, 1.0),
    (1, 0.5),
    (1, 0.25),
    (1, 0.1),
]

In [34]:
for new_rank, layer_keep_fraction in tqdm(rank_keep_fractions):
    key = f"persona_{PERSONA}_rank_{new_rank}_keepfrac_{layer_keep_fraction}"

    all_existing_answers = json.loads(output_answers_json_file.read_text())
    if key in all_existing_answers["answers"].keys():
        print("Skipping, already exists")
        continue

    overwrite_with_downranked_model(model, new_rank=new_rank, layer_keep_fraction=layer_keep_fraction)
    print(f"Generated answers for rank {key}")
    answers = process_messages_batch(model, messages_batch)
    all_existing_answers["answers"][key] = answers
    output_answers_json_file.write_text(json.dumps(all_existing_answers, indent=4))


  0%|          | 0/11 [00:00<?, ?it/s]

restore_lora_modules
get_layers_to_keep
modifying LoRA modules


100%|██████████| 2665/2665 [00:00<00:00, 72656.85it/s]


Generated answers for rank persona_poeticism_rank_64_keepfrac_0.0


  9%|▉         | 1/11 [00:14<02:20, 14.00s/it]

...

 91%|█████████ | 10/11 [2:35:33<13:02, 782.31s/it]

restore_lora_modules
get_layers_to_keep
modifying LoRA modules


100%|██████████| 2665/2665 [02:28<00:00, 17.93it/s]


Generated answers for rank persona_poeticism_rank_1_keepfrac_0.1


100%|██████████| 11/11 [2:38:15<00:00, 863.24s/it]
